# Análisis de Jueces Ciegos — Evaluación de Autenticidad
### EntrevistasCEV — Trabajo de Grado · Sergio Sosa

Análisis descriptivo y tests estadísticos no paramétricos para la **evaluación ciega** de autenticidad de respuestas.

**Diseño experimental:**
- 39 prompts (pares de respuestas de las 4 condiciones: A=baseline, B=bdi_only, C=rag_only, D=bdi_rag)
- 3 jueces LLM: GPT-5, Gemini 3 Flash, Qwen 3.6 Plus
- Escala de autenticidad: 1–10 por condición
- Diseño de bloques (el mismo prompt es evaluado por los 3 jueces)



In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats
from itertools import combinations

# ── Rutas ──────────────────────────────────────────────────────────────────────
_NOTEBOOKS = Path(".").resolve()              # jupyter/notebooks/
_ROOT      = _NOTEBOOKS.parent.parent         # raíz del proyecto
_EVAL      = _ROOT / "src" / "evaluacion"     # src/evaluacion/
XLSX       = _ROOT / "plantilla_anotacion.xlsx"
OUT_DIR    = _EVAL / "promedios" / "analisis_descriptivo"
OUT_DIR.mkdir(parents=True, exist_ok=True)

if not XLSX.exists():
    raise FileNotFoundError(f"No se encontró {XLSX}")

print(f"Excel: {XLSX}")
print(f"Salidas en: {OUT_DIR}")

## 1. Carga y validación de datos

In [ ]:
mapeo_raw = pd.read_excel(XLSX, sheet_name="Mapeo_Referencia", header=None)
mapeo = mapeo_raw.iloc[2:].copy()
mapeo.columns = ["Prompt", "Actor", "A_es", "B_es", "C_es", "D_es"]
mapeo = mapeo[mapeo["Prompt"].str.startswith("Prompt", na=False)]
print(f"Prompts en el mapeo: {len(mapeo)}")

JUDGE_SHEETS = {
    "GPT-5"         : "Resultados_GPT5",
    "Gemini 3 Flash": "Resultados_Gemini3Flash",
    "Qwen 3.6 Plus" : "Resultados_Qwen36Plus",
}

all_scores = []
for judge, sheet in JUDGE_SHEETS.items():
    df = pd.read_excel(XLSX, sheet_name=sheet)
    for _, row in df.iterrows():
        prompt = row["Prompt"]
        m = mapeo[mapeo["Prompt"] == prompt]
        if len(m) == 0:
            continue
        m = m.iloc[0]
        for letter in ["A", "B", "C", "D"]:
            cond  = m[f"{letter}_es"]
            score = row[f"Punt_{letter}"]
            all_scores.append({"juez": judge, "prompt": prompt,
                                "actor": m["Actor"], "condicion": cond, "score": score})

scores_df = pd.DataFrame(all_scores)
print(f"Registros cargados: {len(scores_df)}")
print(f"  Jueces: {scores_df['juez'].unique().tolist()}")
print(f"  Condiciones: {scores_df['condicion'].unique().tolist()}")
print(f"  Prompts únicos: {scores_df['prompt'].nunique()}")
scores_df.head(6)

In [ ]:
CONDITIONS  = ["baseline", "bdi_only", "rag_only", "bdi_rag"]
COND_SHORT  = {"baseline": "BASE", "bdi_only": "BDI", "rag_only": "RAG", "bdi_rag": "BDI+RAG"}
COND_LABELS = {
    "baseline": "BASE\n(sin RAG, sin BDI)",
    "bdi_only" : "BDI\n(BDI, sin RAG)",
    "rag_only" : "RAG\n(RAG, sin BDI)",
    "bdi_rag"  : "BDI+RAG\n(sistema completo)",
}
COLORS = {"baseline": "#d9534f", "bdi_only": "#f0ad4e",
          "rag_only": "#5bc0de", "bdi_rag": "#5cb85c"}
judges = list(JUDGE_SHEETS.keys())

def fmt_p(p):
    return f"{p:.2e}" if p < 0.001 else f"{p:.4f}"

print("Configuración lista.")

## 2. Estadísticos descriptivos por juez × condición

**Unidad de análisis:** n=39 por celda (juez × condición).  
Se reportan medianas e IQR porque los datos son ordinales y no normales (ver Shapiro-Wilk).

In [ ]:
rows = []
for judge in judges:
    for cond in CONDITIONS:
        vals = scores_df[(scores_df["juez"] == judge) & (scores_df["condicion"] == cond)]["score"]
        q1, q3 = vals.quantile(0.25), vals.quantile(0.75)
        stat_sw, p_sw = stats.shapiro(vals.values)
        rows.append({
            "Juez"      : judge,
            "Condicion" : cond,
            "n"         : len(vals),
            "Mediana"   : vals.median(),
            "IQR"       : round(q3 - q1, 2),
            "Media"     : round(vals.mean(), 2),
            "DE"        : round(vals.std(), 2),
            "Min"       : vals.min(),
            "Max"       : vals.max(),
            "W_Shapiro" : round(stat_sw, 4),
            "p_Shapiro" : fmt_p(p_sw),
            "Normal"    : "Sí" if p_sw > 0.05 else "No",
        })

desc_table = pd.DataFrame(rows)
desc_table.to_csv(OUT_DIR / "tabla_descriptivos_por_juez.csv", index=False)

rows_r = []
for cond in CONDITIONS:
    sub = desc_table[desc_table["Condicion"] == cond]
    row = {"Condicion": cond}
    for j in judges:
        s = sub[sub["Juez"] == j].iloc[0]
        row[f"Med ({j})"] = s["Mediana"]
        row[f"IQR ({j})"] = s["IQR"]
    row["Todos no normales"] = "Sí" if (sub["Normal"] == "No").all() else "No"
    rows_r.append(row)

resumen_table = pd.DataFrame(rows_r)
resumen_table.to_csv(OUT_DIR / "tabla_descriptivos_resumen.csv", index=False)

print("=== Tabla de descriptivos (n=39 por juez × condición) ===")
desc_table

In [ ]:
print("=== Resumen por condición ===")
resumen_table

## 3. Test de Levene — Homocedasticidad por juez

In [ ]:
print("=== Levene (homocedasticidad dentro de cada Friedman) ===")
for judge in judges:
    groups_j = [
        scores_df[(scores_df["juez"] == judge) & (scores_df["condicion"] == c)]["score"].values
        for c in CONDITIONS
    ]
    lev_stat, lev_p = stats.levene(*groups_j)
    result = "Varianzas homogéneas" if lev_p > 0.05 else "Varianzas heterogéneas"
    print(f"  {judge}: W={lev_stat:.4f}, p={lev_p:.4f}  →  {result}")

## 4. Histogramas por juez × condición

Justificación de no-parametría: distribución real de scores que entra a cada Friedman (n=39 por celda).

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(16, 12))
fig.suptitle("Distribución de scores por juez y condición (n=39 por celda)",
             fontsize=13, fontweight="bold")

for row_i, judge in enumerate(judges):
    for col_i, cond in enumerate(CONDITIONS):
        ax   = axes[row_i][col_i]
        vals = scores_df[(scores_df["juez"] == judge) & (scores_df["condicion"] == cond)]["score"]
        q1, q3 = vals.quantile(0.25), vals.quantile(0.75)
        stat_sw, p_sw = stats.shapiro(vals.values)

        ax.hist(vals, bins=range(1, 11), align="left", rwidth=0.8,
                color=COLORS[cond], alpha=0.85, edgecolor="white")
        ax.axvline(vals.median(), color="black", linestyle="--",
                   linewidth=1.8, label=f"Med={vals.median():.0f}")

        if col_i == 0:
            ax.set_ylabel(judge, fontsize=9, fontweight="bold")
        if row_i == 0:
            ax.set_title(COND_LABELS[cond], fontsize=9)
        ax.set_xticks(range(1, 10))
        ax.set_xlim(0.5, 9.5)
        ax.legend(fontsize=7, loc="upper left")
        bg = "lightyellow" if p_sw > 0.05 else "#ffe0e0"
        ax.text(0.97, 0.95,
                f"IQR={q3-q1:.0f}\nW={stat_sw:.3f}\np={p_sw:.4f}",
                transform=ax.transAxes, ha="right", va="top", fontsize=7.5,
                bbox=dict(boxstyle="round,pad=0.2", facecolor=bg, alpha=0.9))

plt.tight_layout()
plt.savefig(OUT_DIR / "histogramas_juez_condicion.png", dpi=150, bbox_inches="tight")
plt.show()
print("Guardado: histogramas_juez_condicion.png")

## 5. Boxplots comparativos por condición y juez

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 6), sharey=True)
fig.suptitle("Distribución de scores por condición y juez (n=39 por caja)",
             fontsize=12, fontweight="bold")

for ax, judge in zip(axes, judges):
    data_j = [
        scores_df[(scores_df["juez"] == judge) & (scores_df["condicion"] == c)]["score"].values
        for c in CONDITIONS
    ]
    bp = ax.boxplot(data_j, patch_artist=True,
                    medianprops=dict(color="black", linewidth=2.5),
                    whiskerprops=dict(linewidth=1.5), capprops=dict(linewidth=1.5),
                    flierprops=dict(marker="o", markersize=4, alpha=0.5))
    for patch, cond in zip(bp["boxes"], CONDITIONS):
        patch.set_facecolor(COLORS[cond])
        patch.set_alpha(0.85)

    ax.set_title(f"{judge}\n(n=39 por condición)", fontsize=10, fontweight="bold")
    ax.set_xticks(range(1, 5))
    ax.set_xticklabels([COND_SHORT[c] for c in CONDITIONS], fontsize=9)
    ax.set_ylim(0, 11)
    ax.yaxis.grid(True, linestyle="--", alpha=0.6)
    ax.set_axisbelow(True)
    if ax is axes[0]:
        ax.set_ylabel("Score de autenticidad (1–10)", fontsize=10)

    for i, cond in enumerate(CONDITIONS, 1):
        vals = scores_df[(scores_df["juez"] == judge) & (scores_df["condicion"] == cond)]["score"]
        q1, q3 = vals.quantile(0.25), vals.quantile(0.75)
        ax.text(i, 0.4, f"Med={vals.median():.0f}\nIQR={q3-q1:.0f}",
                ha="center", va="bottom", fontsize=7.5,
                bbox=dict(boxstyle="round,pad=0.2", facecolor="lightyellow", alpha=0.9))

patches = [mpatches.Patch(color=COLORS[c], label=COND_LABELS[c].replace("\n", " ")) for c in CONDITIONS]
fig.legend(handles=patches, loc="lower center", ncol=4, fontsize=9, bbox_to_anchor=(0.5, -0.06))
plt.tight_layout()
plt.savefig(OUT_DIR / "boxplot_por_juez.png", dpi=150, bbox_inches="tight")
plt.show()
print("Guardado: boxplot_por_juez.png")

## 6. Test de Friedman — ¿Difieren las 4 condiciones?

**H₀:** Las 4 condiciones tienen distribuciones idénticas (según cada juez).  
**Diseño de bloques:** cada prompt es un bloque; los 4 scores son las observaciones del bloque.

Usado porque:
- Datos ordinales (scores enteros 1–10)
- No normalidad confirmada (Shapiro-Wilk)
- Diseño de medidas repetidas (mismo prompt → 4 condiciones)

In [ ]:
friedman_rows = []
print("=== Test de Friedman (por juez, n=39 bloques) ===")
print(f"{'Juez':<20} {'χ²':>8} {'p':>10}  {'Significativo?'}")
print("-" * 55)
for judge in judges:
    groups = [
        scores_df[(scores_df["juez"] == judge) & (scores_df["condicion"] == c)]["score"].values
        for c in CONDITIONS
    ]
    stat, p = stats.friedmanchisquare(*groups)
    sig = "SÍ (***)" if p < 0.001 else ("SÍ" if p < 0.05 else "No")
    print(f"{judge:<20} {stat:>8.3f} {p:>10.4f}  {sig}")
    friedman_rows.append({"juez": judge, "chi2": round(stat, 3),
                           "p_valor": round(p, 6), "significativo": p < 0.05})

df_friedman = pd.DataFrame(friedman_rows)
df_friedman.to_csv(OUT_DIR / "friedman.csv", index=False)

## 7. Wilcoxon pares — Post-hoc con corrección de Bonferroni

Aplicado cuando Friedman rechaza H₀. Compara cada par de condiciones.  
**Corrección:** Bonferroni sobre 6 pares → α_ajustado = 0.05 / 6 ≈ 0.0083

In [ ]:
N_PARES   = 6
ALPHA_ADJ = 0.05 / N_PARES

wilcoxon_rows = []
print(f"=== Wilcoxon pares (α ajustado Bonferroni = {ALPHA_ADJ:.4f}) ===")
for judge in judges:
    print(f"\n  {judge}:")
    for c1, c2 in combinations(CONDITIONS, 2):
        g1 = scores_df[(scores_df["juez"] == judge) & (scores_df["condicion"] == c1)]["score"].values
        g2 = scores_df[(scores_df["juez"] == judge) & (scores_df["condicion"] == c2)]["score"].values
        try:
            stat, p = stats.wilcoxon(g1, g2, alternative="two-sided")
        except Exception:
            stat, p = np.nan, np.nan
        sig  = "***" if p < 0.001 else ("**" if p < 0.01 else ("*" if p < ALPHA_ADJ else "ns"))
        med1, med2 = float(np.median(g1)), float(np.median(g2))
        dir_e = f"{COND_SHORT[c1]}>{COND_SHORT[c2]}" if med1 > med2 else f"{COND_SHORT[c2]}>{COND_SHORT[c1]}"
        print(f"    {COND_SHORT[c1]:>7} vs {COND_SHORT[c2]:<7}  "
              f"W={stat:>7.1f}  p={p:.4f}  {sig:<3}  ({dir_e})")
        wilcoxon_rows.append({
            "juez": judge, "cond_A": c1, "cond_B": c2,
            "W_stat": round(float(stat), 2), "p_raw": round(float(p), 6),
            "p_bonferroni": round(min(float(p) * N_PARES, 1.0), 6),
            "significativo": p < ALPHA_ADJ,
            "mediana_A": med1, "mediana_B": med2, "direccion": dir_e,
        })

df_wilcoxon = pd.DataFrame(wilcoxon_rows)
df_wilcoxon.to_csv(OUT_DIR / "wilcoxon_pares.csv", index=False)
df_wilcoxon

## 8. Resumen para la tesis — Justificación de pruebas no paramétricas

In [ ]:
lines = [
    "=== RESUMEN: JUSTIFICACIÓN DE PRUEBAS NO PARAMÉTRICAS ===", "",
    "UNIDAD DE ANÁLISIS:",
    "  Tests (Friedman, Wilcoxon) se ejecutan POR JUEZ con n=39 bloques.",
    "  Descriptivos y Shapiro-Wilk por juez (n=39), NO como pool n=117.", "",
    "1. NATURALEZA DE LOS DATOS",
    "   - Escala: scores enteros [1-10] asignados por LLMs",
    "   - Diseño: 39 prompts × 4 condiciones × 3 jueces (medidas repetidas en bloques)", "",
    "2. ESTADÍSTICOS DESCRIPTIVOS POR JUEZ (n=39)", "",
]
for judge in judges:
    lines.append(f"   {judge}:")
    for cond in CONDITIONS:
        vals = scores_df[(scores_df["juez"] == judge) & (scores_df["condicion"] == cond)]["score"]
        q1, q3 = vals.quantile(0.25), vals.quantile(0.75)
        stat_sw, p_sw = stats.shapiro(vals.values)
        lines.append(f"     {cond:12s}: Med={vals.median():.0f}, IQR={q3-q1:.0f}, "
                     f"Media={vals.mean():.2f}, DE={vals.std():.2f}, "
                     f"Shapiro W={stat_sw:.4f} p={p_sw:.4f}")
    lines.append("")
lines += [
    "3. JUSTIFICACIÓN DE NO-PARAMETRÍA",
    "   a) No normalidad: Shapiro-Wilk rechaza en las 12 distribuciones (p<0.05).",
    "   b) Asimetría opuesta: baseline/bdi_only → masa 3-5; rag_only/bdi_rag → masa 7-9.",
    "   c) Escala acotada y discreta [1-10]: imposible ser gaussiana.",
    "   d) Diseño de bloques: Friedman más apropiado que Kruskal-Wallis.", "",
    "4. RESULTADOS FRIEDMAN",
]
for _, r in df_friedman.iterrows():
    sig = "SÍ significativo" if r["significativo"] else "No significativo"
    lines.append(f"   {r['juez']}: χ²={r['chi2']:.3f}, p={r['p_valor']:.6f}  →  {sig}")

summary = "\n".join(lines)
(OUT_DIR / "justificacion_no_parametrica.txt").write_text(summary, encoding="utf-8")
print(summary)

## Archivos generados

In [ ]:
for f in sorted(OUT_DIR.iterdir()):
    print(f"  ok  {f.name:<50}  {f.stat().st_size/1024:>6.1f} KB")